## Importing Required Libraries and Modules

This cell imports all the libraries and modules required throughout the notebook for building, fine-tuning, training, and evaluating the ResNet-50 brain tumor classification model.

The main components include:

* **`torch` and `torch.nn`** for building and training the neural network.
* **`torchvision.models`** for loading the pretrained ResNet-50 architecture and its associated pretrained weights.
* **`torchvision.datasets` and `torchvision.transforms`** for loading the image dataset and applying preprocessing and data augmentation.
* **`DataLoader`** for efficiently loading images in batches during training and evaluation.
* **`AdamW`** for optimizing the model parameters.
* **Learning-rate schedulers** for controlling how the learning rate changes throughout training.
* **`sklearn.metrics`** for evaluating the trained model using accuracy, precision, recall, F1-score, confusion matrices, and AUC-ROC.
* **`numpy`** for numerical operations.
* **`Path`** for handling dataset and filesystem paths in a platform-independent way.

Importing all required dependencies at the beginning of the notebook keeps the later stages of the fine-tuning pipeline organized and makes it clear which tools are used for each part of the project.


In [103]:
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode

from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.optim import AdamW

from torch.utils.data import DataLoader

from torch import nn
import torch

from pathlib import Path

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score
import numpy as np

## Defining Data Augmentation, Preprocessing, and Data Loaders

Before training the pretrained ResNet-50 model, the brain MRI images must be transformed into a format compatible with the network. This cell defines separate preprocessing pipelines for the training and test datasets, loads the images using `ImageFolder`, and creates data loaders for batch-based training and evaluation.

### Training Transformations

The training pipeline includes both preprocessing and data augmentation:

* **Resize to `(224, 224)`**: ResNet-50 expects images with a spatial resolution compatible with its standard pretrained configuration.
* **Random horizontal flip**: Randomly flips images with a probability of `0.5`, introducing additional variation into the training data.
* **Random rotation**: Rotates images by up to `15` degrees to improve robustness to small orientation changes.
* **Random affine translation**: Slightly shifts images horizontally and vertically, helping the model become less sensitive to small positional differences.
* **Convert to tensor**: Converts images into PyTorch tensors.
* **Normalization**: Uses the mean and standard deviation values from the ImageNet dataset, ensuring that the input distribution is compatible with the pretrained ResNet-50 weights.

These augmentations are applied only during training to improve generalization and reduce the risk of the model memorizing the exact appearance and positioning of the training images.

### Test Transformations

The test pipeline performs only deterministic preprocessing:

* Resize the image to `(224, 224)`.
* Convert it to a PyTorch tensor.
* Normalize it using the ImageNet mean and standard deviation.

No random augmentation is applied to the test set so that evaluation remains consistent and reproducible.

### Dataset and Data Loaders

The datasets are loaded using `ImageFolder`, which automatically assigns class labels based on the directory structure. The `DataLoader` objects then provide the images to the model in batches of `32`.

The training loader uses `shuffle=True` so that the order of training samples changes between epochs, while the test loader provides batches for evaluating the model on previously unseen data.


In [104]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=InterpolationMode.LANCZOS),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=InterpolationMode.LANCZOS),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root='data/train', transform=train_transform)
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

test_dataset = datasets.ImageFolder(root='data/test', transform=test_transform)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=True)

## Initializing the Pretrained ResNet-50 Model and Training Configuration

This cell initializes the pretrained ResNet-50 model and configures the main components required for fine-tuning, including the optimizer, learning-rate scheduler, loss function, computation device, and model checkpoint path.

### Loading and Adapting ResNet-50

The model is initialized using pretrained ImageNet weights:

```python
model = resnet50(weights=ResNet50_Weights.DEFAULT)
```

Rather than training the network from scratch, fine-tuning starts from a model that has already learned useful visual features such as edges, textures, shapes, and higher-level image patterns.

The original ResNet-50 classification layer is then replaced:

```python
model.fc = nn.Linear(model.fc.in_features, 4)
```

The new fully connected layer produces four output values, corresponding to the four brain MRI classes in the dataset. This allows the pretrained feature extractor to be adapted to the specific brain tumor classification task.

### Optimization

The model is optimized using the **AdamW** optimizer with a learning rate of `1e-4` and weight decay of `1e-3`. AdamW is commonly used for fine-tuning deep neural networks because it combines adaptive gradient updates with decoupled weight regularization.

### Learning-Rate Scheduling

Training is divided into two stages:

1. **Warmup Stage — 2 epochs:**
   The learning rate gradually increases from `1%` of the target learning rate to the full learning rate. This helps prevent unstable updates when fine-tuning begins.

2. **Cosine Annealing Stage:**
   After warmup, the learning rate gradually decreases following a cosine curve until reaching a minimum value of `1e-5`. Reducing the learning rate later in training allows the model to make smaller and more refined parameter updates.

These two stages are combined using `SequentialLR`.

### Loss Function and Device Configuration

Since this is a multi-class classification problem, `CrossEntropyLoss` is used to compare the model's predicted class probabilities with the correct class labels.

The model is moved to a CUDA-enabled GPU when available; otherwise, training falls back to the CPU.

Finally, a path is defined for saving the trained model checkpoint.


In [105]:
WARMUP_EPOCHS = 2
MAIN_EPOCHS = 5

model = resnet50(weights=ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 4)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

warmup = LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=WARMUP_EPOCHS,
)

cosine = CosineAnnealingLR(
    optimizer,
    T_max=MAIN_EPOCHS - WARMUP_EPOCHS,
    eta_min=1e-5,
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup, cosine],
    milestones=[WARMUP_EPOCHS],
)

criterion = nn.CrossEntropyLoss()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

path = Path("Model.pth")

## Saving the Training Checkpoint

This function saves the current state of the training process to disk as a PyTorch checkpoint.

The checkpoint contains:

* **Model state:** The learned parameters of the fine-tuned ResNet-50 model.
* **Optimizer state:** The internal state of the AdamW optimizer, allowing training to resume without resetting its adaptive optimization statistics.
* **Scheduler state:** The current state of the learning-rate scheduler, preserving the training schedule when resuming.

Saving these components together makes the checkpoint more complete than saving only the model weights. It allows the training process to be restored from the same state in which it was saved.

The checkpoint is stored at the path defined earlier using `torch.save()`.


In [106]:
def save():
    print("Saving Model...")

    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
    }

    torch.save(checkpoint, path)

## Loading a Previously Saved Training Checkpoint

This function checks whether a previously saved training checkpoint exists and restores the training state when one is found.

If the checkpoint file exists, the function:

* Loads the checkpoint onto the currently selected computation device.
* Restores the fine-tuned ResNet-50 model parameters.
* Restores the AdamW optimizer state.
* Restores the learning-rate scheduler state.

Restoring all three components is important because it allows training to continue from a consistent state rather than loading only the model weights and restarting the optimizer and learning-rate schedule.

If no checkpoint is found, the model continues using its default initialization and training configuration.

The separator printed at the end helps visually distinguish the checkpoint-loading stage from the following training output.


In [107]:
def load():
    if path.exists():
        print(f"Found checkpoint at {path}, Loading model...")

        data = torch.load(path, map_location=device)

        model.load_state_dict(data['model_state_dict'])
        optimizer.load_state_dict(data['optimizer_state_dict'])
        scheduler.load_state_dict(data['scheduler_state_dict'])

    else:
        print(f"No checkpoint found at {path}, Initializing Default Values")

    print(f"============================================================")

## Evaluating Model Performance

This function evaluates the fine-tuned ResNet-50 model on the test dataset using multiple complementary performance metrics.

The model is first switched to evaluation mode using `model.eval()`. This ensures that layers such as dropout and batch normalization behave consistently during evaluation. Gradient computation is also disabled using `torch.no_grad()` because model parameters are not updated during testing, reducing memory usage and computational overhead.

For each batch in the test dataset, the function:

* Performs a forward pass through the model.
* Calculates the cross-entropy loss.
* Converts the model outputs into class probabilities using the softmax function.
* Determines the predicted class with the highest probability.
* Stores the predictions, true labels, and class probabilities for evaluation after all test samples have been processed.

After processing the complete test set, the function calculates several metrics:

### Overall Metrics

* **Test Loss:** Measures the average difference between the predicted outputs and the true labels.
* **Accuracy:** Measures the percentage of correctly classified images.
* **Macro Precision:** Calculates precision independently for each class and then averages the results, giving every class equal importance.
* **Macro Recall:** Measures the average ability of the model to correctly identify samples from each class.
* **Macro F1-Score:** Provides a balanced measure of precision and recall across all classes.

### Per-Class Metrics

Precision, recall, F1-score, and support are calculated separately for each class. This is particularly important because a high overall accuracy can sometimes hide poor performance on individual classes.

### Confusion Matrix

The confusion matrix provides a detailed view of how predictions are distributed across the true classes. It helps identify which classes the model correctly recognizes and which classes are frequently confused with one another.

### AUC-ROC

The multi-class AUC-ROC score is calculated using a **One-vs-Rest (OvR)** strategy with macro averaging. Unlike accuracy, this metric evaluates how well the model's predicted probabilities distinguish each class from the remaining classes.

Finally, the model is returned to training mode using `model.train()` so that the function can be safely called during the training process without affecting subsequent training iterations.


In [108]:
def evaluate():

    model.eval()

    test_loss = 0.0
    all_predictions, all_labels, all_probabilities = [], [], []

    with torch.no_grad():
        for images, labels in test_loader:

            images, labels = images.to(device), labels.to(device)

            output = model(images)

            loss = criterion(output, labels)
            test_loss += loss.item()

            probabilities = torch.softmax(output, dim=1)
            all_probabilities.append(probabilities.cpu().numpy())

            predictions = probabilities.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())

            all_labels.append(labels.cpu().numpy())

    test_loss = test_loss / len(test_loader)

    all_predictions = np.concatenate(all_predictions)
    all_labels = np.concatenate(all_labels)
    all_probabilities = np.concatenate(all_probabilities)

    accuracy = accuracy_score(all_labels, all_predictions)

    precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_predictions, average=None, zero_division=0)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(all_labels, all_predictions, average="macro", zero_division=0)

    matrix = confusion_matrix(all_labels, all_predictions)

    auc = roc_auc_score(all_labels, all_probabilities, multi_class="ovr", average="macro")

    print(f"==================== Performance Report ====================")

    print(f"Test Loss: {test_loss:.4f}")
    print(f"Accuracy:  {(accuracy * 100):.4f}%")

    print()

    print(f"Macro Precision: {precision_macro:.4f}")
    print(f"Macro F1: {f1_macro:.4f}")
    print(f"Macro Recall: {recall_macro:.4f}")

    print()

    print(f"AUC-ROC: {auc:.4f}")

    print()

    print(f"Per-class breakdown:")
    print()
    for i in range(len(precision)):
        print(f"  Class {i}: P={precision[i]:.4f}  R={recall[i]:.4f} F1={f1[i]:.4f}  n={support[i]} ")

    print()

    print(f"Confusion Matrix:\n{matrix}")
    print(f"============================================================")

    model.train()

## Calculating Validation Loss

This function performs a validation pass over the test dataset and calculates the model's average loss without updating any parameters.

The model is first switched to evaluation mode using `model.eval()`, and gradient computation is disabled with `torch.no_grad()`. For each batch, the model performs a forward pass and calculates the cross-entropy loss between its predictions and the true class labels.

The loss values from all batches are accumulated and averaged to produce a single validation loss value:

```python
validation_loss = validation_loss / len(test_loader)
```

Unlike the `evaluate()` function, this function calculates only the validation loss. Its purpose is to provide a lightweight metric that can be called regularly during training to monitor how well the model is generalizing.

After validation is complete, the model is returned to training mode using `model.train()`, and the average validation loss is returned.


In [109]:
def validate():
    model.eval()

    validation_loss = 0
    with torch.no_grad():
        for images, labels in test_loader:

            images, labels = images.to(device), labels.to(device)

            output = model(images)
            loss = criterion(output, labels)

            validation_loss += loss.item()

    validation_loss = validation_loss / len(test_loader)

    model.train()

    return validation_loss

## Training and Fine-Tuning the ResNet-50 Model

This function controls the complete fine-tuning process, including checkpoint loading, forward and backward propagation, parameter updates, learning-rate scheduling, validation, detailed evaluation, and optional checkpoint saving.

The training process begins by attempting to load a previously saved checkpoint. If one exists, the model, optimizer, and learning-rate scheduler states are restored before training continues.

### Training Loop

Training runs for the combined number of warmup and main training epochs. During each epoch, the model processes the training dataset batch by batch.

For every batch:

1. The images and labels are moved to the selected computation device.
2. The model performs a **forward pass** to generate class predictions.
3. The cross-entropy loss is calculated.
4. Previous gradients are cleared using `optimizer.zero_grad()`.
5. **Backpropagation** computes gradients with respect to the model parameters.
6. The AdamW optimizer updates the model parameters using these gradients.
7. The batch loss is accumulated to calculate the average training loss for the epoch.

### Learning-Rate Scheduling

After each epoch, the learning-rate scheduler advances to the next stage of the training schedule. This controls the transition from the initial warmup phase to cosine annealing.

### Validation and Evaluation

Once an epoch is completed, the average training loss and validation loss are calculated. The model is then evaluated using the more comprehensive `evaluate()` function, which reports metrics including accuracy, precision, recall, F1-score, AUC-ROC, per-class performance, and the confusion matrix.

### Checkpoint Saving

After each epoch, the user is given the option to save the current training state. If selected, the model parameters, optimizer state, and scheduler state are stored as a checkpoint.

This design allows the training process to be monitored after every epoch and provides the ability to preserve intermediate model states during experimentation.


In [110]:
def train():
    print(f"Initializing Training Session... ")

    load()
    model.train()

    for epoch in range(WARMUP_EPOCHS + MAIN_EPOCHS):

        train_loss = 0
        for images, labels in train_loader:

            images, labels = images.to(device), labels.to(device)

            output = model(images)

            loss = criterion(output, labels)

            optimizer.zero_grad()
            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        scheduler.step()

        train_loss = train_loss / len(train_loader)
        validation_loss = validate()

        print(f"Epoch: ({epoch + 1}/{WARMUP_EPOCHS + MAIN_EPOCHS})| Train Loss: {train_loss}| Validation Loss: {validation_loss}")

        evaluate()

        user = input("Do You Want To Save The Model? (y/n) ").strip().lower()
        if user == "y":
            save()

        print(f"============================================================")

## Evaluating the Initial Model and Starting Fine-Tuning

The training pipeline is executed in two stages.

First, `evaluate()` is called to measure the performance of the model **before fine-tuning begins**. This provides a baseline that can later be compared with the model's performance after training.

Next, `train()` starts the complete fine-tuning process. The training function attempts to load a previous checkpoint, trains the model for the configured number of epochs, validates and evaluates performance after each epoch, and optionally saves the model state.

Evaluating the model before training makes it possible to observe how much the fine-tuning process improves performance on the brain tumor classification task.


In [ ]:
evaluate()
train()